# PhysIR — Figures & Tables generator (Kaggle, ONE FLOW, **CPU-only**)

Produces every remaining *data-derived* figure and every Chapter-4 table from the stored
artifacts. **Accelerator: None (CPU) is sufficient — ~10–15 min end-to-end, zero GPU quota.** The only
inference is ~6 forward passes for the qualitative grid; the real cost is decoding test images,
which a GPU does not speed up. Attaching **T4 ×2** is harmless (the code auto-detects and uses
CUDA) but buys little here — save that quota for the seed replicates in
`stage6_seed_replicates_kaggle.ipynb`, which genuinely runs ~28 h/arm.

**Resume:** every figure and table is idempotent (existing outputs are skipped) and a restore
cell re-imports a previous version's outputs — so if a session is ever cut short, re-running
picks up exactly what's missing.

| # | Output | file |
|---|---|---|
| 1 | Qualitative crop grid (Fig-4.A) — data-driven image selection | `figures/fig_qualitative_grid.png` |
| 2 | Perception–distortion trajectory (Fig-3.J) | `figures/fig_trajectory.png` |
| 3 | Retinex decomposition panel (Eq. 3.3 operator, our data) | `figures/fig_retinex_decomposition.png` |
| 4 | Oppenheim–Lim phase/amplitude swap panel | `figures/fig_phase_swap.png` |
| 5 | Per-image Δ histogram (the right-tail evidence) | `figures/fig_delta_hist.png` |
| 6 | All Ch.4 tables (CSV + markdown) | `tables/*.csv`, `tables/*.md` |

**NOT generated here** (draw.io / TikZ work): the block diagrams Fig-3.A, 3.C, 3.E, 3.F,
3.H, 3.I. **Not here either:** any training — seed replicates live in the stage6 notebooks.

## Inputs to attach
| dataset | contains | note |
|---|---|---|
| `physir-code` | train.py, archs/, losses/ | already uploaded |
| `lolblur-dataset` | LOLBlur train/test | already uploaded |
| **`physir-artifacts`** | the whole `.metrics/` tree (per-image CSVs, training/perceptual logs, sweep dirs, results JSONs) | **NEW — upload** |
| **`gabrielafotey/physir-models-evaluations-set`** | `DarkIR_384.pt`, `stage4_physir_final_latest.pt`, `stage5_percep_latest.pt`, `stage5_baseline_latest.pt` (+ two per-image CSVs) — nested under `model_evaluations/model_evaluations/` | ✅ uploaded; nesting and slug are auto-resolved |

Kaggle cannot mount Google Drive, hence the two uploads. Clean macOS junk before zipping:
`find . \( -name '._*' -o -name '.DS_Store' \) -delete`

Every cell is idempotent (existing outputs are skipped) and every figure carries **no baked-in
title or caption** — captions belong to the thesis document. Figure sizes follow two-column
publication norms (7.16 in double-column, 3.5 in single-column, 300 dpi).

## 1. Install

In [ ]:
# ptflops/einops/pytorch_msssim are UNGUARDED imports in archs/__init__.py + losses/ —
# without them `import train as T` (needed by the crop grid) fails. lpips/pyiqa are guarded.
!pip install -q lpips pytorch-msssim einops ptflops tabulate 2>/dev/null

import importlib.util as _iu
_need = ['ptflops', 'einops', 'pytorch_msssim', 'torch', 'torchvision',
         'matplotlib', 'pandas', 'numpy', 'PIL']
_missing = [m for m in _need if _iu.find_spec(m) is None]
print('dependency check:', 'all present ✅' if not _missing else f'MISSING {_missing} ❌')
assert not _missing, f'install failed for {_missing} — re-run this cell (Internet must be ON)'

## 2. Inputs, paths, output dirs

In [ ]:
import os, sys, glob, json
import numpy as np

INPUT_BASE = '/kaggle/input'          # (overridden only by the offline self-test)

# ── Attached datasets. These are HINTS: if a slug differs, the marker-based
#    auto-discovery below still finds the right folder, at any nesting depth. ──
HINT_CODE    = 'physir-code'
HINT_LOLBLUR = 'lolblur-dataset'
HINT_ARTIF   = 'physir-artifacts'                    # = the local .metrics tree
HINT_CKPTS   = 'physir-models-evaluations-set'       # gabrielafotey/physir-models-evaluations-set
# ─────────────────────────────────────────────────────────────────────────────
WORK = '/kaggle/working'
FIG  = os.path.join(WORK, 'figures'); os.makedirs(FIG, exist_ok=True)
TAB  = os.path.join(WORK, 'tables');  os.makedirs(TAB, exist_ok=True)

ROOTS = sorted(d for d in glob.glob(os.path.join(INPUT_BASE, '*')) if os.path.isdir(d))

def _dir_containing(marker, hint=None):
    """Deepest-agnostic search: any attached dataset holding <marker>, hint first."""
    ordered = sorted(ROOTS, key=lambda d: (hint or '') not in os.path.basename(d))
    for r in ordered:
        if os.path.exists(os.path.join(r, marker)): return r
        hits = sorted(glob.glob(os.path.join(r, '**', marker), recursive=True))
        if hits: return hits[0][:-len(marker)].rstrip(os.sep) or r
    return None

CODE_ROOT = _dir_containing('train.py', HINT_CODE)
LOLBLUR   = _dir_containing(os.path.join('test', 'low_blur_noise'), HINT_LOLBLUR)
assert CODE_ROOT, f'physir-code not found among {ROOTS}'
assert LOLBLUR,   f'LOLBlur test/ not found among {ROOTS}'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)

# Artifacts + checkpoints may be nested (e.g. model_evaluations/model_evaluations/...)
# and per-image CSVs live in BOTH datasets — so searches span every non-image root.
def _root_for(hint):
    m = [r for r in ROOTS if hint in os.path.basename(r)]
    return m[0] if m else None
MET       = _root_for(HINT_ARTIF) or _dir_containing('per_image', HINT_ARTIF)
CKPT_ROOT = _root_for(HINT_CKPTS) or next(
    (r for r in ROOTS if glob.glob(os.path.join(r, '**', '*.pt'), recursive=True)), None)

SEARCH = [r for r in dict.fromkeys([MET, CKPT_ROOT] + ROOTS)
          if r and r not in (LOLBLUR, CODE_ROOT)]          # skip the image/code trees
# A re-eval output (slug containing 'corrected') always WINS over older per-image CSVs of
# the same name — this is how the shuffle-fix results supersede the shuffled ones.
SEARCH.sort(key=lambda r: 'corrected' not in os.path.basename(r).lower())
if any('corrected' in os.path.basename(r).lower() for r in SEARCH):
    print('NOTE: a "corrected" source is attached and takes precedence for per-image CSVs')

def _is_corrected(path):
    """A per-image CSV written AFTER the shuffle fix carries an `fname` column; that column
    is the proof of stable ordering, so it — not a folder name — decides precedence."""
    try:
        with open(path) as f:
            return 'fname' in (f.readline() or '')
    except Exception:
        return False

def find_csv(name, prefer_corrected=True):
    hits = []
    for r in SEARCH:
        hits += sorted(glob.glob(os.path.join(r, '**', name), recursive=True))
    if not hits:
        return None
    if prefer_corrected and name.startswith('per_image'):
        good = [h for h in hits if _is_corrected(h)]
        if good:
            return good[0]          # a corrected copy always beats a stale one
    return hits[0]

def csv_origin(path):
    """(dataset basename, corrected?) — for the preflight report."""
    if not path: return ('—', False)
    rel = os.path.relpath(path, INPUT_BASE)
    return (rel.split(os.sep)[0], _is_corrected(path))

def find_ckpt(stem):
    for r in SEARCH:
        hits = sorted(h for h in glob.glob(os.path.join(r, '**', stem + '*'), recursive=True)
                      if h.endswith('.pt'))
        if hits: return hits[0]
    return None

print('attached   :', [os.path.basename(r) for r in ROOTS])
print('CODE_ROOT  :', CODE_ROOT)
print('LOLBLUR    :', LOLBLUR)
print('ARTIFACTS  :', MET or 'NOT FOUND')
print('CHECKPOINTS:', CKPT_ROOT or 'NOT ATTACHED (crop grid will be skipped)')
if CKPT_ROOT:
    pts = sorted(os.path.basename(p) for p in
                 glob.glob(os.path.join(CKPT_ROOT, '**', '*.pt'), recursive=True))
    print('  .pt files:', pts)
assert MET, 'physir-artifacts (the .metrics tree) not found'


## 2b. Preflight — verify every required artifact is present

In [ ]:
# ── Preflight: verify physir-artifacts (+ checkpoints) hold everything needed ─
# Non-fatal by default: reports exactly which outputs can be produced and which will be
# skipped, so you can fix the upload BEFORE spending a session. Set STRICT=True to hard-fail.
STRICT   = False
EXPECT_N = 1800          # LOLBlur test set size
PI_COLS  = {'idx', 'psnr', 'ssim', 'lpips'}

import csv as _csv
ARMS = ['abl_baseline','abl_phase','abl_retinex','abl_full',
        'abl_blur','abl_blurnoedge','abl_percep','abl_full_percep']
ANCHORS = ['per_image_stage4_physir_final','per_image_stage5_baseline','per_image_stage5_percep',
           'per_image_physir_full','per_image_abl_percep384','per_image_darkir_m_released']

CORRECTED, STALE = [], []
def check_per_image(fname):
    """→ (status, detail, idx_set). status: 'ok' | 'bad' | 'missing'"""
    p = find_csv(fname)
    if not p: return 'missing', '', None
    try:
        rows = list(_csv.DictReader(open(p)))
    except Exception as e:
        return 'bad', f'unreadable: {e}', None
    if not rows: return 'bad', 'empty', None
    cols = set(rows[0])
    if not PI_COLS <= cols:
        return 'bad', f'columns {sorted(cols)} missing {sorted(PI_COLS-cols)}', None
    src, corrected = csv_origin(p)
    (CORRECTED if corrected else STALE).append(fname)
    tag = '[corrected]' if corrected else '[STALE ⚠]'
    extra = ' +dual-LPIPS' if 'lpips_darkirconv' in cols else ''
    n = len(rows)
    flag = '' if n == EXPECT_N else f'  ⚠ n={n} (expected {EXPECT_N})'
    return 'ok', f'n={n}{extra}{flag}  {tag} ← {src}', {int(r['idx']) for r in rows}

# 1) per-image CSVs -----------------------------------------------------------
print('── per-image CSVs ' + '─'*52)
found, idx_sets = {}, {}
targets  = [f'per_image_{a}_{t}.csv' for a in ARMS for t in ('ep150','last')]
targets += [s + '.csv' for s in ANCHORS]
for f in targets:
    st, detail, ids = check_per_image(f)
    found[f] = st
    if ids: idx_sets[f] = ids
    mark = {'ok':'✅','bad':'❌','missing':'—'}[st]
    if st != 'missing': print(f'  {mark} {f:44s} {detail}')
miss = [f for f,s in found.items() if s == 'missing']
if miss:
    print(f'  — missing ({len(miss)}):')
    for f in miss: print(f'      {f}')

# 1b) ordering provenance -----------------------------------------------------
print('\n── ordering provenance ' + '─'*47)
# Ordering only matters for PAIRED statistics; every mean is order-independent. So a stale
# CSV blocks a comparison ONLY if some paired test actually consumes it.
PAIRED_INPUTS = {f'per_image_{a}_last.csv' for a in ARMS} | {s + '.csv' for s in ANCHORS}
stale_blocking = [f for f in STALE if f in PAIRED_INPUTS]
stale_harmless = [f for f in STALE if f not in PAIRED_INPUTS]
print(f'  corrected (has `fname`, stable order) : {len(CORRECTED)}')
print(f'  stale but MEANS-ONLY (harmless)       : {len(stale_harmless)}'
      + (f'  e.g. {stale_harmless[0]}' if stale_harmless else ''))
print(f'  stale AND used by a paired test       : {len(stale_blocking)}')
if stale_harmless:
    print('    ↳ these feed mean-based tables only (e.g. the @ep150 operating point);')
    print('      means are order-independent, so they are fine as they are.')
if stale_blocking:
    print('  ❌ BLOCKING — re-run reeval_per_image_kaggle.ipynb for:')
    for f in stale_blocking: print(f'      {f}')

# 2) idx alignment (the premise of every paired test) --------------------------
if idx_sets:
    ref_name, ref = next(iter(idx_sets.items()))
    bad = [f for f, s in idx_sets.items() if s != ref]
    print(f'\n── idx alignment ' + '─'*53)
    print(f'  ✅ all {len(idx_sets)} CSVs share the same {len(ref)} indices'
          if not bad else f'  ❌ index mismatch vs {ref_name}: {bad}')

# 3) training / perceptual logs ------------------------------------------------
print('\n── logs ' + '─'*62)
def find_log(name, prefer='stage4'):
    hits = sorted(glob.glob(os.path.join(MET, '**', name), recursive=True))
    pref = [h for h in hits if prefer in h]
    return (pref or hits or [None])[0]
tl, pl = find_log('training_log.csv'), find_log('perceptual_log.csv')
for lab, p, col in (('stage4 training_log.csv', tl, 'val_psnr'),
                    ('stage4 perceptual_log.csv', pl, 'musiq')):
    if not p: print(f'  — {lab:28s} MISSING'); continue
    rows = list(_csv.DictReader(open(p)))
    has  = sum(1 for r in rows if r.get(col))
    print(f'  ✅ {lab:28s} {len(rows):4d} rows, {has} with {col}   ({os.path.relpath(p, MET)})')
sweeps = [d for d in sorted(glob.glob(os.path.join(MET,'**','sweep_*'), recursive=True))
          if os.path.isfile(os.path.join(d,'training_log.csv'))]
print(f'  {"✅" if sweeps else "—"} weight-sweep dirs{"":13s} {len(sweeps)} found')

# 4) results JSONs (optional cross-check of table means) -----------------------
js = sorted(glob.glob(os.path.join(MET,'**','*results*.json'), recursive=True))
print(f'  {"✅" if js else "—"} results JSONs{"":16s} {len(js)} found (optional)')

# 5) checkpoints (crop grid only) ---------------------------------------------
print('\n── checkpoints (Fig-4.A only) ' + '─'*40)
ck_ok = True
for lab, stem in [('DarkIR-m','DarkIR_384'), ('PhysIR','stage4_physir_final')]:
    p = find_ckpt(stem)
    print(f'  {"✅" if p else "❌"} {lab:10s} {os.path.basename(p) if p else "MISSING (" + stem + "*.pt)"}')
    ck_ok &= bool(p)

# 6) verdict -------------------------------------------------------------------
ok = lambda *fs: all(found.get(f) == 'ok' for f in fs)
plan = [
 ('Fig-4.A crop grid',      ck_ok and (ok('per_image_physir_full.csv') or ok('per_image_stage4_physir_final.csv'))
                                   and ok('per_image_darkir_m_released.csv')
                                   and not stale_blocking),   # selection uses paired deltas
 ('Fig-3.J trajectory',     bool(tl and pl)),
 ('Retinex panel',          True),          # needs only LOLBlur images
 ('Phase-swap panel',       True),
 ('Δ histogram',            ok('per_image_stage4_physir_final.csv','per_image_stage5_baseline.csv')
                            and not stale_blocking),      # paired deltas
 ('table_ablation_256 @ep150 rows', True),  # means only — stale ordering is irrelevant here
 ('table_ablation_256',     any(found.get(f'per_image_{a}_last.csv')=='ok' for a in ARMS)),
 ('table_384_and_darkir',   any(found.get(s+'.csv')=='ok' for s in ANCHORS)),
 ('table_wilcoxon_all',     ok('per_image_abl_full_last.csv','per_image_abl_baseline_last.csv')
                            and not stale_blocking),      # paired → needs corrected inputs
 ('table_weight_sweep',     bool(sweeps)),
]
print('\n── verdict ' + '─'*59)
for lab, good in plan:
    print(f'  {"✅ will produce" if good else "⏭  will SKIP  "}  {lab}')
blocked = [l for l, g in plan if not g]
if blocked:
    msg = f'{len(blocked)} output(s) blocked by missing artifacts: {blocked}'
    if STRICT: raise SystemExit('PREFLIGHT FAILED — ' + msg)
    print('\n⚠ ' + msg + '\n  Fix the upload and re-run this cell; everything else still runs.')
else:
    print('\n🎉 preflight clean — every figure and table can be produced.')


## 3. Publication figure style

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# publication defaults: no titles baked in, tight boxes, 300 dpi
plt.rcParams.update({
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.01, 'font.size': 7, 'axes.titlesize': 7,
    'axes.labelsize': 7, 'legend.fontsize': 6, 'xtick.labelsize': 6,
    'ytick.labelsize': 6, 'axes.linewidth': 0.6, 'lines.linewidth': 1.1,
    'font.family': 'DejaVu Sans',
})
W2, W1 = 7.16, 3.5           # double- / single-column widths (inches)
def done(p):
    if os.path.exists(p):
        print('exists, skipping:', os.path.basename(p)); return True
    return False
print('style set | double-col %.2f in, single-col %.2f in' % (W2, W1))

## 3b. Session restore + device check (re-runs skip completed outputs)

In [ ]:
# Continuation sessions: Add Input → Your Work → this notebook (previous version's output),
# then Run All. Already-generated figures/tables are copied back and SKIPPED, so a re-run only
# does what's missing. Fresh sessions print 'nothing to restore'.
import shutil
def _restore():
    n = 0
    for sub, dst_dir in (('figures', FIG), ('tables', TAB)):
        for pat in (f'/kaggle/input/*/{sub}/*', f'/kaggle/input/*/*/{sub}/*'):
            for src in glob.glob(pat):
                dst = os.path.join(dst_dir, os.path.basename(src))
                if os.path.isfile(src) and not os.path.exists(dst):
                    shutil.copy2(src, dst); n += 1
    print(f'restored {n} previously generated file(s)' if n else 'fresh session — nothing to restore')
_restore()

import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEV}  |  CPU is sufficient: expect ~10-15 min end-to-end'
      + ('  (GPU attached — inference is faster but image decoding dominates)' if DEV=='cuda' else ''))

## 4. Fig-4.A — qualitative crop grid (data-driven selection)

In [ ]:
# ── Fig-4.A qualitative crop grid ───────────────────────────────────────────
# Selection is DATA-DRIVEN: rank the 1800 test images by the paired PhysIR-vs-DarkIR-m
# PSNR difference, then take the 95th / 50th / 5th percentile cases (strong win / typical /
# near-parity-or-loss). Identical indices AND identical crop windows for every model.
LABELS_ON   = True     # column headers (Input / DarkIR-m / PhysIR / GT) — keep on
ANNOT_PSNR  = False    # per-crop PSNR text; off keeps the figure clean
CROP        = 192      # crop side in pixels (source res 1120x640)
PCTS        = [95, 50, 5]
OUTP = os.path.join(FIG, 'fig_qualitative_grid.png')

MODELS = [('DarkIR-m', 'DarkIR_384'), ('PhysIR', 'stage4_physir_final')]  # order = columns

if not done(OUTP):
    import csv, torch
    import train as T
    from data import create_test_data

    a = find_csv('per_image_physir_full.csv') or find_csv('per_image_stage4_physir_final.csv')
    b = find_csv('per_image_darkir_m_released.csv')
    assert a and b, 'need PhysIR + DarkIR-m per-image CSVs in physir-artifacts'
    pa = {int(r['idx']): float(r['psnr']) for r in csv.DictReader(open(a))}
    pb = {int(r['idx']): float(r['psnr']) for r in csv.DictReader(open(b))}
    idxs = sorted(set(pa) & set(pb))
    d = np.array([pa[i] - pb[i] for i in idxs])
    picks, seen = [], set()
    for p in PCTS:
        k = int(np.clip(round(np.percentile(np.arange(len(d)), p)), 0, len(d)-1))
        order = np.argsort(d)          # ascending delta
        j = order[k]
        while j in seen:               # avoid duplicates on ties
            k = min(k+1, len(d)-1); j = order[k]
        seen.add(j); picks.append((idxs[j], d[j], p))
    print('selected:', [(i, round(float(dd),3), f'p{pp}') for i, dd, pp in picks])

    ckpts = {n: find_ckpt(s) for n, s in MODELS}
    missing = [n for n, p in ckpts.items() if not p]
    assert not missing, f'checkpoints missing: {missing}'

    NET = {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,
           'middle_blk_num_dec':2,'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],
           'dilations':[1,4,9],'extra_depth_wise':True}
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    def load(p):
        ck = torch.load(p, map_location='cpu', weights_only=False)
        st = ck.get('ema') or ck.get('model') or ck.get('params') if isinstance(ck, dict) else ck
        m = T.build_model(NET).to(dev)
        m.load_state_dict({k.replace('module.',''):v for k,v in st.items()}, strict=False)
        return m.eval()
    nets = {n: load(p) for n, p in ckpts.items()}

    opt = {'name':'LOLBlur','val':{'test_path':os.path.join(LOLBLUR,'test'),'batch_size_test':1},
           'train':{'n_workers':2,'verbose':False}}
    raw,_ = create_test_data(rank=0, world_size=1, opt=opt)
    loader = raw if not isinstance(raw, dict) else list(raw.values())[0]['loader']

    want = {i for i,_,_ in picks}; last = max(want)
    grabbed = {}
    with torch.no_grad():
        for i,(high,low) in enumerate(loader):
            if i in want:
                hi, lo = high.to(dev), low.to(dev)
                preds = {}
                for n, m in nets.items():
                    p = m(lo, side_loss=False).clamp(0,1)
                    preds[n] = p[:,:,:hi.shape[2],:hi.shape[3]]
                # crop window = max GT gradient energy (detail-rich region), same for all models
                g = hi[0].mean(0)
                gy = (g[1:,:]-g[:-1,:]).abs(); gx = (g[:,1:]-g[:,:-1]).abs()
                e = gy[:, :gx.shape[1]] + gx[:gy.shape[0], :]
                H, Wd = e.shape; step = 32; best, bxy = -1, (0,0)
                for y in range(0, max(H-CROP,1), step):
                    for x in range(0, max(Wd-CROP,1), step):
                        s = float(e[y:y+CROP, x:x+CROP].sum())
                        if s > best: best, bxy = s, (y,x)
                y0,x0 = bxy
                to_np = lambda t: t[0].permute(1,2,0).cpu().numpy()
                grabbed[i] = dict(full=to_np(lo), box=(y0,x0),
                                  crops=[('Input', to_np(lo)[y0:y0+CROP, x0:x0+CROP])] +
                                        [(n, to_np(preds[n])[y0:y0+CROP, x0:x0+CROP]) for n,_ in MODELS] +
                                        [('Ground truth', to_np(hi)[y0:y0+CROP, x0:x0+CROP])],
                                  psnrs=[None] + [round(float(20*np.log10(1.0/(np.sqrt(
                                        float(((preds[n][:,:,y0:y0+CROP,x0:x0+CROP]-hi[:,:,y0:y0+CROP,x0:x0+CROP])**2).mean()))+1e-8))),2)
                                        for n,_ in MODELS] + [None])
            if i >= last: break

    import matplotlib.patches as mp
    ncol = 2 + len(MODELS)                     # full | Input | models... | GT  == 3+len
    ncol = 1 + len(grabbed[picks[0][0]]['crops'])
    fig, axes = plt.subplots(len(picks), ncol, figsize=(W2, W2*len(picks)/ncol*0.62))
    for r,(i,dd,pp) in enumerate(picks):
        g = grabbed[i]; y0,x0 = g['box']
        ax = axes[r,0]; ax.imshow(np.clip(g['full'],0,1)); ax.add_patch(
            mp.Rectangle((x0,y0), CROP, CROP, fill=False, lw=0.8, edgecolor='red'))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylabel(f'p{pp}  ({dd:+.2f} dB)', fontsize=6)
        if r==0 and LABELS_ON: ax.set_title('Input (full)', fontsize=7)
        for c,(nm,im) in enumerate(g['crops'], start=1):
            ax = axes[r,c]; ax.imshow(np.clip(im,0,1)); ax.set_xticks([]); ax.set_yticks([])
            if r==0 and LABELS_ON: ax.set_title(nm, fontsize=7)
            if ANNOT_PSNR and g['psnrs'][c-1] is not None:
                ax.text(0.03,0.04,f"{g['psnrs'][c-1]:.2f} dB", transform=ax.transAxes,
                        fontsize=5, color='white', bbox=dict(fc='black', alpha=.5, pad=1, lw=0))
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    fig.savefig(OUTP); plt.close(fig)
    json.dump([{'idx':int(i),'delta_psnr':float(dd),'percentile':int(pp)} for i,dd,pp in picks],
              open(os.path.join(TAB,'fig_qualitative_selection.json'),'w'), indent=1)
    print('wrote', OUTP)

## 5. Fig-3.J — perception-distortion trajectory

In [ ]:
# ── Fig-3.J perception-distortion trajectory (single instance thesis-wide) ──
OUTP = os.path.join(FIG, 'fig_trajectory.png')
if not done(OUTP):
    import csv
    tl = find_csv('training_log.csv'); pl = find_csv('perceptual_log.csv')
    # prefer the stage4 final run's logs
    for cand in sorted(glob.glob(os.path.join(MET,'**','training_log.csv'), recursive=True)):
        if 'stage4' in cand: tl = cand
    for cand in sorted(glob.glob(os.path.join(MET,'**','perceptual_log.csv'), recursive=True)):
        if 'stage4' in cand: pl = cand
    assert tl and pl, 'stage4 logs not found'
    # Resumes re-train the epochs logged after the last checkpoint (log every epoch,
    # checkpoint every 2-5), so a training_log can carry duplicate epoch rows at each resume
    # boundary. The LAST row for an epoch is authoritative — later epochs continue from it.
    def series(path, cols):
        keep = {}
        for r in csv.DictReader(open(path)):
            if all(r.get(c) for c in cols):
                keep[int(r['epoch'])] = [float(r[c]) for c in cols]   # later row wins
        xs = sorted(keep)
        return xs, [list(v) for v in zip(*[keep[x] for x in xs])]
    ep, (ps, lp) = series(tl, ('val_psnr', 'val_lpips'))
    pe, (mu,)    = series(pl, ('musiq',))

    fig, ax = plt.subplots(figsize=(W1, W1*0.68))
    ax.plot(ep, ps, color='#1f77b4', label='PSNR (dB)')
    ax.set_xlabel('epoch'); ax.set_ylabel('PSNR (dB)', color='#1f77b4')
    ax.tick_params(axis='y', labelcolor='#1f77b4')
    # two markers: the PSNR peak AND the reported model — the contrast IS the policy argument
    pk = int(np.argmax(ps))
    for xv, lab in ((ep[pk], 'PSNR peak'), (ep[-1], 'reported model')):
        ax.axvline(xv, ls='--', lw=0.6, color='grey')
        ax.annotate(lab, xy=(xv, ax.get_ylim()[0]), xytext=(2, 3),
                    textcoords='offset points', fontsize=5.5, color='grey',
                    rotation=90, va='bottom', ha='left')   # off the curves, along the axis
    ax2 = ax.twinx()
    ax2.plot(ep, lp, color='#d62728', label='LPIPS ↓')
    ax2.plot(pe, [m/max(mu)*max(lp) for m in mu], color='#2ca02c', ls='-.', lw=0.9,
         label='MUSIQ (scaled ↑)')
    ax2.set_ylabel('LPIPS  /  MUSIQ (scaled)', fontsize=6)
    h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc='center right', frameon=False)
    fig.savefig(OUTP); plt.close(fig)
    print('wrote', OUTP, f'| PSNR peak ep {ep[pk]}, reported model ep {ep[-1]}')
    print('caption note: LPIPS and MUSIQ share a scaled right axis; only direction is meaningful.')

## 6. Retinex decomposition panel

In [ ]:
# ── Retinex decomposition panel — Eq. (3.3)/(3.4) operator on our own data ───
# Ch.3-guide figure-status path (a): the panels must MATCH the section's formulation.
#   • computed on a CROP of the training size — a 15x15 maxpool on a full 1120x640 frame is
#     not what the loss sees, and is why an earlier version's illumination showed object
#     structure instead of a smooth field;
#   • reflectance shown in the LOG domain, which is what L_recon actually compares
#     (and which tames the noise amplification of a naive I/L division).
CROP_SIZE  = 256      # training crop; 384 to match the final regime
CROP_XY    = None     # None = auto (most detail-rich region by GT gradient energy)
R_CLIP_PCT = (1, 99)  # percentile window for display scaling of log R
FRAME_IDX  = 0
OUTP = os.path.join(FIG, 'fig_retinex_decomposition.png')

if not done(OUTP):
    import torch, torch.nn.functional as F
    from PIL import Image
    lows  = sorted(glob.glob(os.path.join(LOLBLUR,'test','low_blur_noise','*','*.png')))
    highs = sorted(glob.glob(os.path.join(LOLBLUR,'test','high_sharp_scaled','*','*.png')))
    to_t = lambda p: torch.from_numpy(np.array(Image.open(p).convert('RGB'), np.float32)/255).permute(2,0,1)[None]
    x, t = to_t(lows[FRAME_IDX]), to_t(highs[FRAME_IDX])

    if CROP_XY is None:                      # pick a detail-rich window from the reference
        g = t[0].mean(0)
        e = (g[1:,:-1]-g[:-1,:-1]).abs() + (g[:-1,1:]-g[:-1,:-1]).abs()
        H, W = e.shape; best, CROP_XY = -1, (0,0)
        for yy in range(0, max(H-CROP_SIZE,1), 32):
            for xx in range(0, max(W-CROP_SIZE,1), 32):
                s = float(e[yy:yy+CROP_SIZE, xx:xx+CROP_SIZE].sum())
                if s > best: best, CROP_XY = s, (yy,xx)
    y0, x0 = CROP_XY
    x = x[:, :, y0:y0+CROP_SIZE, x0:x0+CROP_SIZE]
    t = t[:, :, y0:y0+CROP_SIZE, x0:x0+CROP_SIZE]

    EPS = 1e-4
    def illum(z, k=15):                                        # Eq. (3.3)
        m = z.max(dim=1, keepdim=True).values
        return F.max_pool2d(m, k, stride=1, padding=k//2).clamp_min(EPS)
    L   = illum(x)
    logR = torch.log(x + EPS) - torch.log(L + EPS)             # Eq. (3.4)

    def show_log(r):
        v = r[0].permute(1,2,0).numpy()
        lo, hi = np.percentile(v, R_CLIP_PCT[0]), np.percentile(v, R_CLIP_PCT[1])
        return np.clip((v - lo) / (hi - lo + 1e-8), 0, 1)

    panels = [('Input  I',                        np.clip(x[0].permute(1,2,0).numpy(),0,1)),
              ('Illumination  L  (Eq. 3.3)',      np.clip(L[0,0].numpy(),0,1)),
              ('log R = log I − log L  (Eq. 3.4)', show_log(logR)),
              ('Reference  T',                    np.clip(t[0].permute(1,2,0).numpy(),0,1))]
    fig, axes = plt.subplots(1, len(panels), figsize=(W2, W2/len(panels)*1.02))
    for ax,(nm,im) in zip(axes, panels):
        ax.imshow(im, cmap=('gray' if im.ndim==2 else None), vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(nm, fontsize=6.5)
    plt.subplots_adjust(wspace=0.02)
    fig.savefig(OUTP); plt.close(fig)
    print('wrote', OUTP, f'| {CROP_SIZE}px crop @{CROP_XY}, log-domain, 15x15 maxpool')
    print('caption note: log R is displayed on a percentile-scaled grey ramp; the loss compares'
          ' log R_pred against log R_target derived from T by the same operator.')

## 7. Oppenheim-Lim phase/amplitude swap panel

In [ ]:
# ── Oppenheim-Lim amplitude/phase swap panel ────────────────────────────────
IDX_A, IDX_B = 0, 900       # two distinct GT frames
OUTP = os.path.join(FIG, 'fig_phase_swap.png')
if not done(OUTP):
    import torch
    from PIL import Image
    highs = sorted(glob.glob(os.path.join(LOLBLUR,'test','high_sharp_scaled','*','*.png')))
    def load_gray(p):
        return torch.from_numpy(np.array(Image.open(p).convert('L'), np.float32)/255)
    A, B = load_gray(highs[IDX_A]), load_gray(highs[IDX_B])
    h = min(A.shape[0], B.shape[0]); w = min(A.shape[1], B.shape[1])
    A, B = A[:h,:w], B[:h,:w]
    FA, FB = torch.fft.rfft2(A), torch.fft.rfft2(B)
    mix1 = torch.fft.irfft2(FA.abs() * torch.exp(1j*FB.angle()), s=(h,w))   # |A| + phase(B)
    mix2 = torch.fft.irfft2(FB.abs() * torch.exp(1j*FA.angle()), s=(h,w))   # |B| + phase(A)
    norm = lambda z: np.clip((z-z.min())/(z.max()-z.min()+1e-8), 0, 1)
    panels = [('(a) image A', A.numpy()), ('(b) image B', B.numpy()),
              ('(c) |A| + phase(B)', norm(mix1.numpy())), ('(d) |B| + phase(A)', norm(mix2.numpy()))]
    fig, axes = plt.subplots(1, 4, figsize=(W2, W2/4*0.60))
    for ax,(nm,im) in zip(axes, panels):
        ax.imshow(im, cmap='gray'); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(nm, fontsize=7)
    plt.subplots_adjust(wspace=0.02)
    fig.savefig(OUTP); plt.close(fig)
    print('wrote', OUTP, '| (c) reads as B, (d) reads as A -> structure lives in phase')

## 8. Per-image delta histogram

In [ ]:
# ── Per-image delta histogram: the right-tail evidence ──────────────────────
OUTP = os.path.join(FIG, 'fig_delta_hist.png')
if not done(OUTP):
    import csv
    def col(f, c='psnr'):
        return {int(r['idx']): float(r[c]) for r in csv.DictReader(open(f))}
    pairs = [('PhysIR - baseline', 'per_image_stage4_physir_final.csv', 'per_image_stage5_baseline.csv'),
             ('PhysIR - DarkIR-m', 'per_image_physir_full.csv', 'per_image_darkir_m_released.csv')]
    fig, axes = plt.subplots(1, 2, figsize=(W2, W2*0.30))
    for ax,(lab,fa,fb) in zip(axes, pairs):
        A, B = find_csv(fa), find_csv(fb)
        if not (A and B):
            ax.set_visible(False); print('skip (missing):', lab); continue
        a, b = col(A), col(B)
        d = np.array([a[i]-b[i] for i in sorted(set(a)&set(b))])
        ax.hist(d, bins=60, color='#4c72b0', edgecolor='none')
        ax.axvline(0, color='grey', lw=0.6)
        ax.axvline(d.mean(), color='#c44e52', lw=0.9, label=f'mean {d.mean():+.2f}')
        ax.axvline(np.median(d), color='#55a868', lw=0.9, ls='--', label=f'median {np.median(d):+.2f}')
        ax.set_xlabel(f'{lab}  (per-image PSNR, dB)'); ax.set_ylabel('images')
        ax.legend(frameon=False)
    plt.subplots_adjust(wspace=0.25)
    fig.savefig(OUTP); plt.close(fig)
    print('wrote', OUTP)

## 9. All Chapter-4 tables (CSV + markdown)

In [ ]:
# ── All Chapter-4 tables, machine-generated (CSV + markdown) ────────────────
import csv, math, glob as _g
import pandas as pd

METRIC_COLS = ('psnr', 'ssim', 'lpips', 'lpips_darkirconv')

def load_pi(path):
    """Per-image records keyed by idx. NOTE: `fname` holds only the basename, and LOLBlur test
    has 30 clips x 60 frames with repeating basenames (1800 rows -> 90 unique names), so fname
    must NOT be used as a key. It is used below to VERIFY that two files enumerate the same
    images in the same order."""
    rows = list(csv.DictReader(open(path)))
    recs = {int(r['idx']): {m: float(r[m]) for m in METRIC_COLS if r.get(m) not in (None, '')}
            for r in rows}
    order = [r.get('fname') for r in rows] if rows and rows[0].get('fname') else None
    return recs, order

def load_pair(pa, pb, label=''):
    """Join on idx; if both files carry fname, assert element-wise agreement so a mismatched
    ordering can never again pass silently."""
    a, oa = load_pi(pa); b, ob = load_pi(pb)
    if oa and ob and oa != ob:
        n_bad = sum(1 for x, y in zip(oa, ob) if x != y)
        raise SystemExit(f'ORDER MISMATCH in "{label}": {n_bad}/{len(oa)} positions differ — '
                         'these two CSVs enumerate different images per row; re-run the re-eval.')
    if not (oa and ob):
        print(f'  ⚠ {label}: one input lacks fname — alignment assumed, not verified')
    return a, b

def wilcoxon(a, b, key, lower_better=False):
    idx = sorted(set(a)&set(b)); n0 = len(idx)
    d = np.array([a[i][key]-b[i][key] for i in idx])
    if lower_better: d = -d
    d = d[d != 0]; n = len(d)
    if n == 0: return dict(n=n0, median=0.0, pct=50.0, z=0.0, p=1.0, r=0.0)
    order = np.argsort(np.abs(d)); ranks = np.empty(n)
    srt = np.abs(d)[order]; i = 0
    while i < n:                                     # average tied ranks
        j = i
        while j+1 < n and srt[j+1] == srt[i]: j += 1
        ranks[order[i:j+1]] = (i+j)/2 + 1; i = j+1
    W = ranks[d > 0].sum()
    mu = n*(n+1)/4; sd = math.sqrt(n*(n+1)*(2*n+1)/24)
    z = (W-mu)/sd if sd else 0.0
    p = math.erfc(abs(z)/math.sqrt(2))
    return dict(n=n0, median=float(np.median(d)), pct=100*float((d>0).mean()),
                z=float(z), p=float(p), r=abs(z)/math.sqrt(n0))

def stars(p): return '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'ns'

def to_md(df):
    try:
        return df.to_markdown(index=False)          # needs tabulate
    except Exception:                                # hand-rolled fallback
        cols = list(df.columns)
        out = ['| ' + ' | '.join(map(str, cols)) + ' |',
               '|' + '|'.join(['---']*len(cols)) + '|']
        for _, r in df.iterrows():
            out.append('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
        return '\n'.join(out)

def save(df, stem, note=''):
    df.to_csv(os.path.join(TAB, stem+'.csv'), index=False)
    with open(os.path.join(TAB, stem+'.md'), 'w') as f:
        if note: f.write(f'<!-- {note} -->\n')
        f.write(to_md(df))
    print('wrote', stem, f'({len(df)} rows)')

# 1) 256/200 ablation table, both operating points
ARMS = ['abl_baseline','abl_phase','abl_retinex','abl_full','abl_blur',
        'abl_blurnoedge','abl_percep','abl_full_percep']
rows = []
for arm in ARMS:
    for tag in ('ep150','last'):
        f = find_csv(f'per_image_{arm}_{tag}.csv')
        if not f: continue
        d = pd.read_csv(f)
        rows.append(dict(arm=arm, ckpt=tag, psnr=d.psnr.mean(), ssim=d.ssim.mean(),
                         lpips=d.lpips.mean(), n=len(d)))
if rows: save(pd.DataFrame(rows).round(4), 'table_ablation_256')

# 2) 384 anchors + released DarkIR-m (dual LPIPS convention where present)
rows = []
for lab, stem in [('PhysIR (full)','per_image_stage4_physir_final'),
                  ('baseline','per_image_stage5_baseline'),
                  ('abl_percep','per_image_stage5_percep'),
                  ('PhysIR (re-eval)','per_image_physir_full'),
                  ('abl_percep (re-eval)','per_image_abl_percep384'),
                  ('DarkIR-m released','per_image_darkir_m_released')]:
    f = find_csv(stem+'.csv')
    if not f: continue
    d = pd.read_csv(f)
    r = dict(model=lab, psnr=d.psnr.mean(), ssim=d.ssim.mean(), lpips=d.lpips.mean(), n=len(d))
    if 'lpips_darkirconv' in d: r['lpips_darkir_conv'] = d.lpips_darkirconv.mean()
    rows.append(r)
if rows: save(pd.DataFrame(rows).round(4), 'table_384_and_darkir',
              'LPIPS column = [-1,1] scaling (lpips spec); lpips_darkir_conv = unscaled [0,1], their convention')

# 3) the full Wilcoxon suite
PAIRS = [
 ('full vs baseline @384','per_image_stage4_physir_final','per_image_stage5_baseline'),
 ('full vs percep @384','per_image_stage4_physir_final','per_image_stage5_percep'),
 ('percep vs baseline @384','per_image_stage5_percep','per_image_stage5_baseline'),
 ('PhysIR vs DarkIR-m released','per_image_physir_full','per_image_darkir_m_released'),
 ('percep-repro vs DarkIR-m','per_image_abl_percep384','per_image_darkir_m_released'),
 ('full vs baseline @256','per_image_abl_full_last','per_image_abl_baseline_last'),
 ('phase vs baseline @256','per_image_abl_phase_last','per_image_abl_baseline_last'),
 ('retinex vs baseline @256','per_image_abl_retinex_last','per_image_abl_baseline_last'),
 ('percep vs baseline @256','per_image_abl_percep_last','per_image_abl_baseline_last'),
 ('blur vs baseline @256','per_image_abl_blur_last','per_image_abl_baseline_last'),
 ('full vs phase @256 (retinex marginal)','per_image_abl_full_last','per_image_abl_phase_last'),
 ('full vs retinex @256 (phase marginal)','per_image_abl_full_last','per_image_abl_retinex_last'),
 ('full vs percep @256','per_image_abl_full_last','per_image_abl_percep_last'),
 ('full+percep vs full @256','per_image_abl_full_percep_last','per_image_abl_full_last'),
 ('blur-instead-of-edge vs edge+blur','per_image_abl_blurnoedge_last','per_image_abl_blur_last'),
]
rows = []
for lab, fa, fb in PAIRS:
    A, B = find_csv(fa+'.csv'), find_csv(fb+'.csv')
    if not (A and B): print('skip (missing):', lab); continue
    a, b = load_pair(A, B, lab)
    _n = len(set(a) & set(b))
    if _n < 0.99 * max(len(a), len(b)):
        print(f'  ⚠ {lab}: only {_n} of {max(len(a),len(b))} records join — check inputs')
    for k, low in (('psnr',False), ('ssim',False), ('lpips',True)):
        w = wilcoxon(a, b, k, lower_better=low)
        rows.append(dict(comparison=lab, metric=k, median_delta=round(w['median'],4),
                         pct_A_better=round(w['pct'],1), z=round(w['z'],2),
                         p=f"{w['p']:.2e}", sig=stars(w['p']), effect_r=round(w['r'],3), n=w['n']))
if rows: save(pd.DataFrame(rows), 'table_wilcoxon_frame_level_SECONDARY',
              'SECONDARY / frame-level only. The LOLBlur test set is 30 clips x 60 adjacent frames, '
              'so frames are NOT independent and these p-values are inflated by pseudo-replication. '
              'The PRIMARY statistics are clip-level with cluster-bootstrap CIs, produced by '
              'clustered_analysis.py (.metrics/cluster_aware_table.md). Improvement = A - B; '
              'LPIPS sign-flipped so + = better; rows joined by idx with element-wise fname verification.')
print('\n  NOTE: this table is the SECONDARY frame-level column only.')
print('  PRIMARY = clip-level + bootstrap CIs from clustered_analysis.py (run offline).')

# 4) Stage-3 weight sweep (noise-floor support)
rows = []
for d in sorted(_g.glob(os.path.join(MET,'**','sweep_*'), recursive=True)):
    lg = os.path.join(d,'training_log.csv')
    if not os.path.isfile(lg): continue
    best, ep = -1, None
    for r in csv.DictReader(open(lg)):
        if r.get('val_psnr') and float(r['val_psnr'])>best: best, ep = float(r['val_psnr']), int(r['epoch'])
    rows.append(dict(config=os.path.basename(d), best_val_psnr=round(best,4), at_epoch=ep))
if rows: save(pd.DataFrame(rows).sort_values('best_val_psnr', ascending=False), 'table_weight_sweep')

## 10. Bundle outputs

In [ ]:
# ── bundle everything for one download ──────────────────────────────────────
import shutil
shutil.make_archive(os.path.join(WORK,'physir_figures_tables'), 'zip', WORK, 'figures')
print('\nfigures/'); [print('  ', f) for f in sorted(os.listdir(FIG))]
print('tables/');  [print('  ', f) for f in sorted(os.listdir(TAB))]
print('\nDownload physir_figures_tables.zip (figures) + the tables/ folder from the Output tab.')
print('Save a version to persist them.')